# OneVoice V2 — Fine-tune ASR tiếng Việt từ WAV V1

Notebook này train checkpoint ASR thật từ toàn bộ WAV **clean và noisy** của split train. Split dev chỉ để chọn checkpoint; test tuyệt đối không dùng khi train.

Checkpoint tạo ra là Whisper-Tiny adaptation dùng cho bài nộp, không phải thay thế GIPFormer ONNX production.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/OneVoice')
DRIVE = Path('/content/drive/MyDrive')
DATASET_ROOT = DRIVE / 'onevoice_audio_v1'
MANIFEST = DATASET_ROOT / 'manifest.jsonl'
OUTPUT = DRIVE / 'OneVoice/models/whisper_tiny_vi_construction_v1'
os.environ['HF_HOME'] = str(DRIVE / 'OneVoice/model_cache/huggingface')
os.environ['PYTHONUNBUFFERED'] = '1'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Platypus27-coder/OneVoice.git', str(REPO)], check=True)
os.chdir(REPO)
if not MANIFEST.is_file(): raise FileNotFoundError(f'Missing {MANIFEST}; run colab_data_audit_v2.ipynb first.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers==4.57.1', 'tokenizers==0.22.1', 'accelerate>=1.0.0', 'soundfile', 'librosa'], check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Enable T4 GPU: Runtime → Change runtime type → T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0), '| Output:', OUTPUT)


In [ ]:
# Build examples from all valid clean + noisy files.  Test is deliberately excluded.
import json
rows = [json.loads(x) for x in MANIFEST.read_text(encoding='utf-8').splitlines() if x.strip()]
def examples(split):
    result = []
    for row in rows:
        if row.get('split') != split or not str(row.get('text', '')).strip(): continue
        for folder, key in (('clean', 'clean_audio'), ('noisy', 'audio')):
            path = DATASET_ROOT / folder / str(row[key])
            if not path.is_file(): raise FileNotFoundError(path)
            result.append((str(path), str(row['text']).strip()))
    return result
train_rows, dev_rows = examples('train'), examples('dev')
assert train_rows and dev_rows
print(f'train: {len(train_rows)} WAVs (clean + noisy); dev: {len(dev_rows)} WAVs; test used in training: 0')


In [ ]:
# One epoch visits every clean + noisy train WAV once.  Rerun after interruption to resume checkpoint-*.
from dataclasses import dataclass
from typing import Any
import librosa
from torch.utils.data import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.trainer_utils import get_last_checkpoint

MODEL = 'openai/whisper-tiny'
processor = WhisperProcessor.from_pretrained(MODEL, language='Vietnamese', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language='vi', task='transcribe')
model.config.use_cache = False

class AudioDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        path, text = self.rows[i]
        audio, _ = librosa.load(path, sr=16000, mono=True)
        return {'input_features': processor.feature_extractor(audio, sampling_rate=16000).input_features[0], 'labels': processor.tokenizer(text).input_ids}

@dataclass
class Collator:
    processor: Any
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad([{'input_features': x['input_features']} for x in features], return_tensors='pt')
        labels = self.processor.tokenizer.pad([{'input_ids': x['labels']} for x in features], return_tensors='pt')
        batch['labels'] = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        return batch

args = Seq2SeqTrainingArguments(output_dir=str(OUTPUT), per_device_train_batch_size=8, per_device_eval_batch_size=8, gradient_accumulation_steps=2, learning_rate=1e-5, num_train_epochs=1, fp16=True, gradient_checkpointing=True, eval_strategy='steps', eval_steps=500, save_strategy='steps', save_steps=500, save_total_limit=2, logging_steps=25, report_to=[], remove_unused_columns=False, dataloader_num_workers=2, seed=1337)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=AudioDataset(train_rows), eval_dataset=AudioDataset(dev_rows), data_collator=Collator(processor), processing_class=processor.feature_extractor)
resume = get_last_checkpoint(str(OUTPUT)) if OUTPUT.exists() else None
print('Resume:', resume or 'new run')
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(str(OUTPUT / 'best'))
processor.save_pretrained(str(OUTPUT / 'best'))
(OUTPUT / 'run_manifest.json').write_text(json.dumps({'status':'TRAINED_SYNTHETIC_V1','base_model':MODEL,'train_wavs':len(train_rows),'dev_wavs':len(dev_rows),'sources':['clean','noisy'],'test_used_for_training':False,'epochs':1,'caveat':'Synthetic V1 Whisper adaptation; not a GIPFormer ONNX or real-site claim.'}, ensure_ascii=False, indent=2), encoding='utf-8')
print('DONE:', OUTPUT / 'best')
